<a href="https://colab.research.google.com/github/harshsinghdev-11/machine_learning/blob/main/gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 47.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.1
    Uninstalling transformers-5.10.1:
      Successfully uninstalled transformers-5.10.1


In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForMultimodalLM
from transformers import AutoModelForCausalLM
import torch
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-4-12B",
    device_map="auto",
    attn_implementation="sdpa"
)
processor = AutoProcessor.from_pretrained(
    "google/gemma-4-12B",
    padding_side="left"
)

config.json:   0%|          | 0.00/4.38k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/23.9G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-12B-it"

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)


processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.5k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/23.9G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [ ]:
story1 = """ There's a boy in the picture who has a sitar in his hand. He might be thinking in different directions:
Who first thought that a wire could produce such beautiful sounds?
If this thing (the sitar) is not new to him, then he might be thinking about which sur/taal he should play today. He should be asking questions to himself about the last time he played it and analyzing that part of the last time.
He is sharply noticing the bottom part of the sitar.
As his eyes linger on the instrument, his hand rests thoughtfully under his chin, anchoring him in deep concentration. The initial curiosity about the origins of music transitions into a quiet resolve; he realizes that the instrument before him is a bridge between past mistakes and future melodies. Breaking his gaze from the bottom part of the sitar, he gently adjusts his posture, ready to let go of the analytical doubts of his last practice session. He decides to let his fingers find the answer to today's sur, trusting that the simple tension of a wire, which had mystified him moments ago, will once again speak for him. """

In [ ]:
analysisForm = "Find the plot , structured-unstructured , real-bizarre-appropriate and fantasies-imagination in the given story. Story : {" + story1 + "}"

In [ ]:
messages = [
    {
        "role": "user",
        "content": story1,
    },
]
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

resultOfAnalysisForm = model.generate(**inputs, max_new_tokens=200)
print(processor.decode(resultOfAnalysisForm[0][input_len:], skip_special_tokens=False))

KeyboardInterrupt: 

In [ ]:
HeroContext = """ Analysis of the content firstly involves, discrimination of the character with which the subject has identified himself Like (a) the character in whom the story-teller was apparently most interested, whose point of view was adopted, whose feelings and motives ham been most intimately portrayed He for she') is usually (b) the one who most resembles the subject an individual of the same sex of about the same age, status, or role, who shares some of the subject sentiments and aims. This character is called as Hero (either male or female) who plays the leading role in the drama
The interpreter should be prepared to deal with some complication: (1) the identification of subject with character sometimes shifts during the course of the story, there is a sequence of heroes (first, second, etc.) (2) Two forces of the subject's personality may be represented by different characters, for example, an antisocial drive by a criminal and conscience by a law practitioner Here we can say any endopsychic thema (internal dramatic situation) with two component heroes (3) The subject may tell a story that contains a story, such as one in which the hero observes or hears about events in which another character (for whom he feels some sympathy) is leadingly involved. Here we would say of a primary and secondary hero. Then (4), the subject may identify with a character of the opposite sex and express a part of his personality. Finally, there may be no perceptible single hero, either (5) heroship is divided among a number of equally significant, equally differentiated partial heroes (e.g. a group of people), or (6) the chief character (hero) obviously belongs to the object side of the subject-object situation, he is not a component of the story-teller's personality but an element of his environment. The subject, in other words, has not identified with the principal character. The subject himself is not represented by a minor character (hero in our sense). The characterization of the heroes by the interpreter should include superiority (power, ability), inferiority, criminality, mental abnormality, solitariness, belongingness and leadership.  """

In [ ]:
heroIdentification = f"""Identify the Hero and Type of Content in the given story.

Story:
{story1}

Provide the output as valid JSON in this exact format:
{{
  "Hero": "result_hero",
  "Type of Content": "content type from the story",
  "Reasoning":"provide reasoning of the hero identification and type of content"
}}

Instructions for identifying the Hero:
{HeroContext}"""

In [ ]:
def modelPrediction(stepPrompt):
  messages = [
      {
          "role":"user",
          "content":stepPrompt
      }
  ]
  text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    )
  inputs = processor(text=text, return_tensors="pt").to(model.device)
  input_len = inputs["input_ids"].shape[-1]
  resultOfHero = model.generate(**inputs, max_new_tokens=200)
  print(processor.decode(resultOfHero[0][input_len:], skip_special_tokens=False))

In [ ]:
thema = "Find the thema and interpersonal relations of the given story. Story : {" + story1 + "}" + """ provide the output in json format like {
Theme:{
  "need/press":"result",
  "outcome":"outcome result"
  }
  InterpersonalRelations:{
  "result"
  }
} """

In [ ]:
analysisOfNeedsContext = """ A need may express itself subjectively as an impulse, a wish or an
intention or objectively as a trend of overt behavior. One need may function merely as an
instrumental force, subsidiary to the satisfaction of another domination need.The strength of
each variety of need and emotion manifested by the hero is rated on a 1to 5 scale. Where 5 being
the highest possible mark for any variable on a single story The criteria of strength are intensity,
 duration, frequency and importance in the plot. The slightest suggestion of a variable (eg. a flash
 of irritability) is given a mark of 1. whereas an intense form (eg violent anger) or the continued
 or repeated occurrence of a milder form (eg. constant quarreling) is scored 5. Marks 2, 3, and 4
 are given for intermediate intensities of expression."""

In [ ]:
analysisOfNeedsContextAttribute = {
    "Abasement":"To submit to coercion or control in order to avoid blame. punishment, pain or death. To suffer a disagreeable press (insult, injury, defeat) without opposition. To confess, apologize, promise to do better, atone, reform. To resign himself passively to scarcely bearable conditions. Masochism",
    "Achievement":"To work at something important with energy and persistence. To strive to accomplish something creditable. To get ahead in business, to persuade or lead a group, to create something. Ambition manifested in action.",
}

In [ ]:
analysisOfNeeds = f"""
Analysis of needs and rate the need on the basis of thier intensity , duration and frequency.
 """

In [ ]:
for key,value in analysisOfNeedsContextAttribute.items():
  print(key + " : ")
  modelPromptforAnalysis = f"""
  Analysis of needs and rate the need on the basis of thier intensity , duration and frequency.
  How to analyse the need : {analysisOfNeedsContext}
  Need is : {key}
  Description of need : {value}
  Respond ONLY in this exact JSON format, no explanation:
  {{
  "need": "{key}",
  "intensity": <1-5>,
  "duration": <1-5>,
  "frequency": <1-5>,
  "overall": <average of the three, rounded to 1 decimal>
  }}"""
  modelPrediction(modelPromptforAnalysis)

Abasement : 


ValueError: Cannot use apply_chat_template because this processor does not have a chat template.

In [ ]:
tatAttribute = {
    "Hero":heroIdentification
}

In [ ]:
for key,value in tatAttribute.items():
  print(key + " : ")
  modelPrediction(value)

Hero : 
```json
{
  "Hero": "The boy with the sitar",
  "Type of Content": "Internal Monologue/Reflection or Descriptive Narrative",
  "Reasoning": "The story focuses entirely on the internal thoughts, observations, and emotional transition of the boy holding the sitar. The narrative delves into his curiosity about music's origins, self-analysis of his previous practice, and his final decision to trust his instincts to play. This intense focus on the character's thoughts, feelings, and psychological journey makes it an Internal Monologue or Reflection piece. The 'Hero' is clearly this boy, as his perspective and internal conflict drive the entire text. The content is descriptive in how it portrays his mental state and the emotional significance of the instrument."
}
```<turn|>
